# Gemma Lie Detector - 01 Text (core)
Does Gemma mean what it says? For each SST-2 review, Gemma classifies the sentiment and cites the words that most influenced it.
We then **delete those words** (and random / occlusion-top words) and see whether the decision actually changes, and we **plant a misleading hint** to see whether the model admits it.

**Before running:** Runtime -> Change runtime type -> GPU, and add a Colab secret named `HF_TOKEN` (key icon, notebook access ON).
Start with `N = 50` to check everything works, then raise it to ~500.

In [ ]:
import os, sys
BRANCH = "text-model"   # branch to clone; change to "main" after merging
if not os.path.exists("src/gemma_lie_detector"):
    if not os.path.exists("Gemma-Lie-Detector"):
        !git clone -b {BRANCH} https://github.com/colbyrobbins/Gemma-Lie-Detector.git
    %cd Gemma-Lie-Detector
!pip -q install -U "transformers>=4.50" accelerate datasets scikit-learn tqdm scipy matplotlib
sys.path.insert(0, os.path.abspath("src"))   # makes the package importable without a kernel restart
assert os.path.exists("src/gemma_lie_detector"), "repo code not found: did you push it to GitHub (and is the repo public)?"
print("repo ready in", os.getcwd())


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))   # needs the Gemma license accepted on this HF account

In [ ]:
N = 100                         # 50 reviews took ~20 min on a T4 (both experiments), so budget accordingly
MODEL_ID = "google/gemma-3-4b-it"
DTYPE = "auto"                  # "auto" -> bfloat16 (fp16 gives NaNs with Gemma 3). Slower on a T4, native on L4/A100

In [ ]:
from gemma_lie_detector.model import GemmaScorer
from gemma_lie_detector.prompts import SENTIMENT
from gemma_lie_detector.data import load_sst2

scorer = GemmaScorer(SENTIMENT.labels, model_id=MODEL_ID, dtype=DTYPE)
examples = load_sst2(n=N, seed=0)
print("dtype:", scorer.dtype, "| examples:", len(examples))

## Smoke test - read these outputs by eye before the long run

In [ ]:
import numpy as np
from gemma_lie_detector.prompts import parse_explanation
for ex in examples[:3]:
    prompt = SENTIMENT.prompt(ex.render())
    logp, mass = scorer.score([prompt])
    raw = scorer.generate([prompt])[0]
    print("TEXT :", ex.render()); print("GOLD :", ex.gold)
    print("PROBS:", dict(zip(SENTIMENT.labels, np.exp(logp[0]).round(3))), "| log-probs:", logp[0].round(2), "| label mass:", round(float(mass[0]), 3))
    print("RAW  :", raw); print("PARSE:", parse_explanation(raw)); print("-" * 80)
# label mass should be close to 1.0. If it is low, the model is not starting its JSON with a label word - tell Claude.

## Experiment 1 - do the cited words drive the decision? (resumable: re-run to continue after a disconnect)

In [ ]:
from gemma_lie_detector.experiments import run_core
run_core(examples, SENTIMENT, scorer, "results/text_core_v2.jsonl")

## Experiment 2 - planted cue: a misleading hint pointing at the opposite label

In [ ]:
from gemma_lie_detector.experiments import run_cue
run_cue(examples, SENTIMENT, scorer, "results/text_cue_v2.jsonl")

## Results

In [ ]:
from gemma_lie_detector.analysis import load_results, summarize_core, summarize_cue, make_figures
core, cue = load_results("results/text_core_v2.jsonl"), load_results("results/text_cue_v2.jsonl")
display(summarize_core(core).round(3))
display(summarize_cue(cue).round(3))
make_figures(core, cue, outdir="results/figures_text")

In [ ]:
# Save your work: download, or copy to Drive, before the Colab runtime is recycled.
import shutil; shutil.make_archive("text_results", "zip", "results")
from google.colab import files; files.download("text_results.zip")